# Topic Modeling Example

This notebook demonstrates **Latent Dirichlet Allocation (LDA)** for topic modeling using scikit-learn.

Topic modeling is an unsupervised NLP technique that discovers hidden thematic structure in a collection of documents.

**Pipeline:**
1. Sample text corpus
2. Text preprocessing (lowercase, stopword removal)
3. TF-IDF / Count vectorization
4. LDA model training
5. Topic inspection & document-topic distribution
6. Visualization

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import re

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.decomposition import LatentDirichletAllocation

print('Libraries loaded.')

## 1. Sample Corpus

20 short documents covering three hidden topics: **technology**, **health/medicine**, and **sports**.

In [ ]:
documents = [
    # Technology
    "Machine learning algorithms can identify patterns in large datasets automatically.",
    "Neural networks are inspired by the structure of the human brain and process data in layers.",
    "Cloud computing allows companies to store and access data over the internet.",
    "Artificial intelligence is transforming industries from finance to healthcare.",
    "Deep learning models require large amounts of labeled training data.",
    "Software engineers write code to build applications for web and mobile platforms.",
    "Cybersecurity protects computer systems from digital attacks and data breaches.",
    # Health / Medicine
    "Doctors recommend regular exercise and a balanced diet to maintain good health.",
    "Vaccines train the immune system to recognize and fight specific viruses.",
    "High blood pressure increases the risk of heart disease and stroke.",
    "Clinical trials test new drugs and therapies before they are approved for patients.",
    "Mental health is as important as physical health and should not be neglected.",
    "Surgeons use minimally invasive techniques to reduce recovery time for patients.",
    "Nutrition research shows that a Mediterranean diet reduces inflammation.",
    # Sports
    "The football team scored three goals in the second half to win the championship.",
    "Athletes train daily to improve their strength, speed, and endurance.",
    "The Olympic Games bring together thousands of athletes from around the world.",
    "Basketball players need excellent coordination and teamwork to win matches.",
    "The tennis player won the Grand Slam tournament after five intense sets.",
    "Swimming is an excellent full-body workout and a competitive Olympic sport.",
]

print(f'Corpus size: {len(documents)} documents')

## 2. Text Preprocessing

In [ ]:
def preprocess(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    return text

clean_docs = [preprocess(d) for d in documents]

# Show a sample
for orig, clean in zip(documents[:3], clean_docs[:3]):
    print(f'  Original : {orig}')
    print(f'  Cleaned  : {clean}')
    print()

## 3. Vectorization (Bag-of-Words)

LDA works on raw term counts. We remove English stop words.

In [ ]:
vectorizer = CountVectorizer(stop_words='english', min_df=1, max_df=0.95)
dtm = vectorizer.fit_transform(clean_docs)   # document-term matrix
vocab = vectorizer.get_feature_names_out()

print(f'Document-term matrix shape: {dtm.shape}  (docs x vocab)')
print(f'Vocabulary size: {len(vocab)}')

## 4. LDA Model Training

We tell LDA to find **3 topics** (matching our corpus design).

In [ ]:
N_TOPICS = 3

lda = LatentDirichletAllocation(
    n_components=N_TOPICS,
    max_iter=20,
    learning_method='batch',
    random_state=42,
)

lda.fit(dtm)
print('LDA training complete.')
print(f'Log-likelihood: {lda.score(dtm):.1f}')

## 5. Inspect Topics — Top Words per Topic

In [ ]:
def print_top_words(model, feature_names, n_top=10):
    for idx, topic in enumerate(model.components_):
        top_idx = topic.argsort()[-n_top:][::-1]
        top_words = [feature_names[i] for i in top_idx]
        print(f'Topic {idx}: {", ".join(top_words)}')

print_top_words(lda, vocab)

## 6. Document-Topic Distribution

In [ ]:
doc_topics = lda.transform(dtm)   # shape: (n_docs, n_topics)

labels = ['Technology'] * 7 + ['Health'] * 7 + ['Sports'] * 6
dominant_topic = doc_topics.argmax(axis=1)

df = pd.DataFrame({
    'document': [d[:55] + '...' for d in documents],
    'true_label': labels,
    'dominant_topic': dominant_topic,
    **{f'topic_{i}_prob': doc_topics[:, i].round(3) for i in range(N_TOPICS)}
})

pd.set_option('display.max_colwidth', 60)
df

## 7. Visualization

### 7a. Heatmap — Document × Topic probabilities

In [ ]:
fig, ax = plt.subplots(figsize=(7, 9))
im = ax.imshow(doc_topics, aspect='auto', cmap='YlOrRd')

ax.set_xticks(range(N_TOPICS))
ax.set_xticklabels([f'Topic {i}' for i in range(N_TOPICS)])
ax.set_yticks(range(len(documents)))
ax.set_yticklabels([d[:45] + '...' for d in documents], fontsize=7)
ax.set_title('Document–Topic Probability Distribution', fontsize=12)

plt.colorbar(im, ax=ax, label='Probability')
plt.tight_layout()
plt.savefig('doc_topic_heatmap.png', dpi=120, bbox_inches='tight')
plt.show()

### 7b. Bar chart — Top 10 words per topic

In [ ]:
fig, axes = plt.subplots(1, N_TOPICS, figsize=(14, 4), sharey=False)
colors = ['steelblue', 'tomato', 'seagreen']

for idx, (ax, topic) in enumerate(zip(axes, lda.components_)):
    top_n = 10
    top_idx = topic.argsort()[-top_n:][::-1]
    top_words = [vocab[i] for i in top_idx]
    top_vals  = topic[top_idx]
    top_vals  = top_vals / top_vals.sum()   # normalise for display

    ax.barh(top_words[::-1], top_vals[::-1], color=colors[idx])
    ax.set_title(f'Topic {idx}', fontsize=12)
    ax.set_xlabel('Relative weight')

plt.suptitle('Top 10 Words per Topic', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('topic_words_barchart.png', dpi=120, bbox_inches='tight')
plt.show()

### 7c. Stacked bar — per-document topic mix

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
bar_colors = ['steelblue', 'tomato', 'seagreen']
bottom = np.zeros(len(documents))

for i in range(N_TOPICS):
    ax.bar(range(len(documents)), doc_topics[:, i], bottom=bottom,
           label=f'Topic {i}', color=bar_colors[i])
    bottom += doc_topics[:, i]

# Vertical separators between ground-truth groups
ax.axvline(6.5, color='black', linestyle='--', linewidth=1)
ax.axvline(13.5, color='black', linestyle='--', linewidth=1)
ax.text(3,  1.02, 'Technology', ha='center', fontsize=9)
ax.text(10, 1.02, 'Health',     ha='center', fontsize=9)
ax.text(16, 1.02, 'Sports',     ha='center', fontsize=9)

ax.set_xlabel('Document index')
ax.set_ylabel('Topic probability')
ax.set_title('Topic Mixture per Document')
ax.legend(loc='lower right')
plt.tight_layout()
plt.savefig('doc_topic_stacked.png', dpi=120, bbox_inches='tight')
plt.show()

## 8. Predicting Topics for New Documents

In [ ]:
new_docs = [
    "The new GPU accelerates training of deep neural networks significantly.",
    "Eating vegetables and fruits every day keeps the doctor away.",
    "The marathon runner broke the world record at the city championship.",
]

new_dtm = vectorizer.transform([preprocess(d) for d in new_docs])
new_topics = lda.transform(new_dtm)

topic_names = {0: 'Topic 0', 1: 'Topic 1', 2: 'Topic 2'}  # relabel after inspection

for doc, probs in zip(new_docs, new_topics):
    dominant = probs.argmax()
    print(f'Doc : "{doc[:60]}..."')
    print(f'  Probs : {probs.round(3)}  -->  Dominant topic: {dominant}')
    print()